# QD Study Results Viewer

In [1]:
import json
import os
from pathlib import Path
from collections import defaultdict

import pandas as pd

pd.set_option("display.max_colwidth", 50)
pd.set_option("display.max_rows", 100)

In [2]:
# --- Configuration ---
LOGS_DIR = Path("../logs/aira-dojo")
SOLVER = "greedy"  # Change to "evo" for evo results
ISSUE_ID = f"QD_STUDY_{SOLVER}_gdm"

print(f"Looking for runs matching: *{ISSUE_ID}*")
print(f"Logs directory: {LOGS_DIR.resolve()}")

Looking for runs matching: *QD_STUDY_greedy_gdm*
Logs directory: /share/edc/home/antonis/qd-search/aira-dojo/logs/aira-dojo


In [3]:
def load_run(run_dir: Path) -> dict:
    """Load all available data from a single run directory."""
    run = {"dir": run_dir, "dir_name": run_dir.name}

    # Config
    config_path = run_dir / "dojo_config.json"
    config = json.loads(config_path.read_text())
    run["task"] = config.get("task", {}).get("name", "unknown")
    run["seed"] = int(run_dir.name.split("seed_")[1].split("_")[0])
    run["solver"] = config.get("solver", {}).get("_target_", "").split(".")[-1]

    # Journal
    journal_path = run_dir / "json" / "JOURNAL.jsonl"
    entries = [json.loads(l) for l in journal_path.read_text().strip().split("\n")]
    run["total_steps"] = len(entries)
    run["buggy"] = sum(1 for e in entries if e["data"].get("is_buggy"))
    run["good"] = run["total_steps"] - run["buggy"]
    metrics = [e["data"]["metric"] for e in entries if e["data"].get("metric") is not None]
    run["best_journal_metric"] = max(metrics) if metrics else None
    run["journal"] = entries

    # State (timing)
    state_path = run_dir / "json" / "STATE.jsonl"
    if state_path.exists():
        state_lines = state_path.read_text().strip().split("\n")
        last_state = json.loads(state_lines[-1])
        run["running_time_s"] = last_state["data"].get("running_time", 0)
        run["running_time_min"] = round(run["running_time_s"] / 60, 1)
    else:
        run["running_time_s"] = None
        run["running_time_min"] = None

    # Grading report
    report_path = run_dir / "results" / "grading_report.json"
    if report_path.exists():
        report = json.loads(report_path.read_text())
        run["score"] = report.get("score")
        run["gold"] = report.get("gold_medal", False)
        run["silver"] = report.get("silver_medal", False)
        run["bronze"] = report.get("bronze_medal", False)
        run["above_median"] = report.get("above_median", False)
        run["valid_submission"] = report.get("valid_submission", False)
        if run["gold"]:
            run["medal"] = "GOLD"
        elif run["silver"]:
            run["medal"] = "SILVER"
        elif run["bronze"]:
            run["medal"] = "BRONZE"
        elif run["above_median"]:
            run["medal"] = "above_median"
        else:
            run["medal"] = "below_median"
    else:
        run["score"] = None
        run["medal"] = None
        run["valid_submission"] = False

    return run

In [4]:
def load_all_runs(logs_dir: Path, issue_id: str) -> list[dict]:
    """Load all runs matching the issue ID."""
    runs = []
    pattern_dir = logs_dir / f"user_antonis_issue_{issue_id}"
    if not pattern_dir.exists():
        print(f"Directory not found: {pattern_dir}")
        return runs
    for d in sorted(pattern_dir.iterdir()):
        if not d.is_dir():
            continue
        if not (d / "dojo_config.json").exists():
            continue
        runs.append(load_run(d))
    print(f"Loaded {len(runs)} total run directories")
    return runs


def deduplicate_runs(runs: list[dict]) -> list[dict]:
    """For each (task, seed), keep the run with the best score.
    If no run has a score, keep the one with the most steps."""
    grouped = defaultdict(list)
    for r in runs:
        grouped[(r["task"], r["seed"])].append(r)

    best_runs = []
    for key, group in sorted(grouped.items()):
        scored = [r for r in group if r["score"] is not None]
        if scored:
            best = max(scored, key=lambda r: r["score"])
        else:
            best = max(group, key=lambda r: r["total_steps"])
        best_runs.append(best)
    return best_runs


all_runs = load_all_runs(LOGS_DIR, ISSUE_ID)
runs = deduplicate_runs(all_runs)
print(f"After deduplication: {len(runs)} unique (task, seed) runs")

Loaded 20 total run directories
After deduplication: 15 unique (task, seed) runs


In [5]:
# --- Summary Table ---
rows = []
for r in runs:
    rows.append({
        "task": r["task"],
        "seed": r["seed"],
        "steps": r["total_steps"],
        "good": r["good"],
        "buggy": r["buggy"],
        "score": r["score"],
        "medal": r["medal"],
        "time_min": r["running_time_min"],
    })

df = pd.DataFrame(rows).sort_values(["task", "seed"]).reset_index(drop=True)
print(f"\n=== {SOLVER.upper()} Results ===")
df


=== GREEDY Results ===


,task,seed,steps,good,buggy,score,medal,time_min
0,dog-breed-identification,1,30,0,30,NaN,None,40.1
1,dog-breed-identification,2,30,0,30,NaN,None,42.6
2,dog-breed-identification,3,30,0,30,NaN,None,50.7
3,learning-agency-lab-automated-essay-scoring-2,1,30,0,30,NaN,None,178.0
4,learning-agency-lab-automated-essay-scoring-2,2,30,0,30,NaN,None,179.8
5,learning-agency-lab-automated-essay-scoring-2,3,30,0,30,0.82234,below_median,681.1
6,spooky-author-identification,1,30,0,30,NaN,None,31.7
7,spooky-author-identification,2,30,0,30,NaN,None,51.4
8,spooky-author-identification,3,29,0,29,0.43852,below_median,158.5
9,stanford-covid-vaccine,1,30,0,30,0.23440,GOLD,161.3


In [6]:
# --- Pivot: Score by task x seed ---
score_pivot = df.pivot(index="task", columns="seed", values="score")
score_pivot["mean"] = score_pivot.mean(axis=1)
print("\n=== Scores (task x seed) ===")
score_pivot


=== Scores (task x seed) ===


seed,1,2,3,mean
task,,,,
dog-breed-identification,NaN,NaN,NaN,NaN
learning-agency-lab-automated-essay-scoring-2,NaN,NaN,0.82234,0.822340
spooky-author-identification,NaN,NaN,0.43852,0.438520
stanford-covid-vaccine,0.23440,0.23613,NaN,0.235265
tabular-playground-series-dec-2021,0.96302,0.96313,NaN,0.963075


In [7]:
# --- Pivot: Medal by task x seed ---
medal_pivot = df.pivot(index="task", columns="seed", values="medal")
print("\n=== Medals (task x seed) ===")
medal_pivot


=== Medals (task x seed) ===


seed,1,2,3
task,,,
dog-breed-identification,None,None,None
learning-agency-lab-automated-essay-scoring-2,None,None,below_median
spooky-author-identification,None,None,below_median
stanford-covid-vaccine,GOLD,GOLD,None
tabular-playground-series-dec-2021,GOLD,GOLD,None


In [8]:
# --- Aggregate stats ---
print(f"\n=== Aggregate Stats ({SOLVER.upper()}) ===")
total_runs = len(df)
has_score = df["score"].notna().sum()
medals = df["medal"].value_counts()
print(f"Total runs: {total_runs}")
print(f"Runs with valid score: {has_score}/{total_runs}")
print(f"\nMedal distribution:")
for m, count in medals.items():
    print(f"  {m}: {count}")
print(f"\nMean score (where available): {df['score'].mean():.5f}")
print(f"Mean time per run: {df['time_min'].mean():.1f} min")


=== Aggregate Stats (GREEDY) ===
Total runs: 15
Runs with valid score: 6/15

Medal distribution:
  GOLD: 4
  below_median: 2

Mean score (where available): 0.60959
Mean time per run: 163.5 min
